In [2]:
%pip install qiskit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 4.0 MB/s eta 0:00:00


In [3]:
# Day 9: Transpilation and Backend Optimization

import qiskit
from qiskit import QuantumCircuit, transpile

print("Qiskit version:", qiskit.__version__)
print("Imports successful!")
print("Day 9 environment is ready.")

Qiskit version: 2.6.0
Imports successful!
Day 9 environment is ready.


In [4]:
# Step 2: Create the original 3-qubit circuit

qc = QuantumCircuit(3)

qc.h(0)
qc.cx(0, 2)
qc.t(1)
qc.cx(1, 2)
qc.h(2)

print("Original quantum circuit:")
print(qc.draw())

print("\nNumber of qubits:", qc.num_qubits)
print("Circuit depth:", qc.depth())
print("Gate counts:", qc.count_ops())

Original quantum circuit:
     ┌───┐               
q_0: ┤ H ├──■────────────
     ├───┤  │            
q_1: ┤ T ├──┼────■───────
     └───┘┌─┴─┐┌─┴─┐┌───┐
q_2: ─────┤ X ├┤ X ├┤ H ├
          └───┘└───┘└───┘

Number of qubits: 3
Circuit depth: 4
Gate counts: OrderedDict({'h': 2, 'cx': 2, 't': 1})


In [5]:
# Step 3: Transpile with optimization level 0

transpiled_0 = transpile(
    qc,
    optimization_level=0
)

print("Transpiled circuit — Optimization Level 0")
print(transpiled_0.draw())

print("\nOriginal depth:", qc.depth())
print("Transpiled depth:", transpiled_0.depth())

print("\nOriginal gate counts:", qc.count_ops())
print("Transpiled gate counts:", transpiled_0.count_ops())

Transpiled circuit — Optimization Level 0
     ┌───┐               
q_0: ┤ H ├──■────────────
     ├───┤  │            
q_1: ┤ T ├──┼────■───────
     └───┘┌─┴─┐┌─┴─┐┌───┐
q_2: ─────┤ X ├┤ X ├┤ H ├
          └───┘└───┘└───┘

Original depth: 4
Transpiled depth: 4

Original gate counts: OrderedDict({'h': 2, 'cx': 2, 't': 1})
Transpiled gate counts: OrderedDict({'h': 2, 'cx': 2, 't': 1})


In [6]:
# Step 4: Compare all optimization levels

results = []

for level in range(4):
    transpiled = transpile(
        qc,
        optimization_level=level
    )

    results.append({
        "Optimization Level": level,
        "Depth": transpiled.depth(),
        "Total Gates": sum(transpiled.count_ops().values()),
        "Gate Counts": dict(transpiled.count_ops())
    })

for result in results:
    print(f"\nOptimization Level: {result['Optimization Level']}")
    print("Depth:", result["Depth"])
    print("Total gates:", result["Total Gates"])
    print("Gate counts:", result["Gate Counts"])


Optimization Level: 0
Depth: 4
Total gates: 5
Gate counts: {'h': 2, 'cx': 2, 't': 1}

Optimization Level: 1
Depth: 4
Total gates: 5
Gate counts: {'h': 2, 'cx': 2, 't': 1}

Optimization Level: 2
Depth: 4
Total gates: 5
Gate counts: {'h': 2, 'cx': 2, 't': 1}

Optimization Level: 3
Depth: 4
Total gates: 5
Gate counts: {'h': 2, 'cx': 2, 't': 1}


In [7]:
# Step 6: Create a simple linear hardware target

from qiskit.transpiler import CouplingMap

coupling_map = CouplingMap([
    [0, 1],
    [1, 0],
    [1, 2],
    [2, 1]
])

print("Hardware connectivity:")
print(coupling_map)
print("\nNumber of physical qubits:", coupling_map.size())
print("Is q0 directly connected to q2?",
      coupling_map.distance(0, 2) == 1)

Hardware connectivity:
[(0, 1), (1, 0), (1, 2), (2, 1)]

Number of physical qubits: 3
Is q0 directly connected to q2? False


In [8]:
# Step 7: Route the circuit for our linear hardware

routed_qc = transpile(
    qc,
    coupling_map=coupling_map,
    initial_layout=[0, 1, 2],
    optimization_level=0,
    seed_transpiler=42
)

print("Routed circuit:")
print(routed_qc.draw())

print("\nOriginal depth:", qc.depth())
print("Routed depth:", routed_qc.depth())

print("\nOriginal gate counts:", qc.count_ops())
print("Routed gate counts:", routed_qc.count_ops())

print("\nNumber of SWAP gates:",
      routed_qc.count_ops().get("swap", 0))

Routed circuit:
         ┌───┐                  
q_0 -> 0 ┤ H ├─────■────────────
         ├───┤   ┌─┴─┐┌───┐┌───┐
q_1 -> 1 ┤ T ├─X─┤ X ├┤ X ├┤ H ├
         └───┘ │ └───┘└─┬─┘└───┘
q_2 -> 2 ──────X────────■───────
                                

Original depth: 4
Routed depth: 5

Original gate counts: OrderedDict({'h': 2, 'cx': 2, 't': 1})
Routed gate counts: OrderedDict({'h': 2, 'cx': 2, 't': 1, 'swap': 1})

Number of SWAP gates: 1


In [9]:
# Step 8: Measure routing overhead

original_gates = sum(qc.count_ops().values())
routed_gates = sum(routed_qc.count_ops().values())

original_depth = qc.depth()
routed_depth = routed_qc.depth()

print("Added gates:", routed_gates - original_gates)
print("Added depth:", routed_depth - original_depth)

print(
    "Gate-count increase (%):",
    round((routed_gates - original_gates) / original_gates * 100, 2)
)

print(
    "Depth increase (%):",
    round((routed_depth - original_depth) / original_depth * 100, 2)
)

Added gates: 1
Added depth: 1
Gate-count increase (%): 20.0
Depth increase (%): 25.0


In [10]:
# Step 9: Visual comparison

print("=" * 45)
print("ORIGINAL CIRCUIT")
print("=" * 45)
print(qc.draw())

print("\n" + "=" * 45)
print("ROUTED CIRCUIT")
print("=" * 45)
print(routed_qc.draw())

ORIGINAL CIRCUIT
     ┌───┐               
q_0: ┤ H ├──■────────────
     ├───┤  │            
q_1: ┤ T ├──┼────■───────
     └───┘┌─┴─┐┌─┴─┐┌───┐
q_2: ─────┤ X ├┤ X ├┤ H ├
          └───┘└───┘└───┘

ROUTED CIRCUIT
         ┌───┐                  
q_0 -> 0 ┤ H ├─────■────────────
         ├───┤   ┌─┴─┐┌───┐┌───┐
q_1 -> 1 ┤ T ├─X─┤ X ├┤ X ├┤ H ├
         └───┘ │ └───┘└─┬─┘└───┘
q_2 -> 2 ──────X────────■───────
                                


In [11]:
# Step 10: Verify quantum-operation equivalence

from qiskit.quantum_info import Operator
import numpy as np

original_operator = Operator(qc)
routed_operator = Operator(routed_qc)

equivalent = np.allclose(
    original_operator.data,
    routed_operator.data
)

print("Are the original and routed operations equivalent?")
print(equivalent)

Are the original and routed operations equivalent?
False


In [12]:
# Step 11: Inspect the transpiler's layout

print("Transpiler layout information:")
print(routed_qc.layout)

if routed_qc.layout is not None:
    print("\nInitial layout:")
    print(routed_qc.layout.initial_layout)

    print("\nFinal layout:")
    print(routed_qc.layout.final_layout)

Transpiler layout information:
TranspileLayout(initial_layout=Layout({
0: <Qubit register=(3, "q"), index=0>,
1: <Qubit register=(3, "q"), index=1>,
2: <Qubit register=(3, "q"), index=2>
}), input_qubit_mapping={<Qubit register=(3, "q"), index=0>: 0, <Qubit register=(3, "q"), index=1>: 1, <Qubit register=(3, "q"), index=2>: 2}, final_layout=Layout({
0: <Qubit register=(3, "q"), index=0>,
2: <Qubit register=(3, "q"), index=1>,
1: <Qubit register=(3, "q"), index=2>
}), _input_qubit_count=3, _output_qubit_list=[<Qubit register=(3, "q"), index=0>, <Qubit register=(3, "q"), index=1>, <Qubit register=(3, "q"), index=2>])

Initial layout:
Layout({
0: <Qubit register=(3, "q"), index=0>,
1: <Qubit register=(3, "q"), index=1>,
2: <Qubit register=(3, "q"), index=2>
})

Final layout:
Layout({
0: <Qubit register=(3, "q"), index=0>,
2: <Qubit register=(3, "q"), index=1>,
1: <Qubit register=(3, "q"), index=2>
})


In [13]:
# Step 12: Compare operations with layout information

from qiskit.quantum_info import Operator
import numpy as np

routed_operator_corrected = Operator.from_circuit(
    routed_qc,
    ignore_set_layout=False
)

original_operator = Operator(qc)

print(
    "Equivalent after layout-aware conversion:",
    np.allclose(
        original_operator.data,
        routed_operator_corrected.data
    )
)

Equivalent after layout-aware conversion: True


In [14]:
# Step 13: Compare routing optimization levels

routed_level_0 = transpile(
    qc,
    coupling_map=coupling_map,
    initial_layout=[0, 1, 2],
    optimization_level=0,
    seed_transpiler=42
)

routed_level_3 = transpile(
    qc,
    coupling_map=coupling_map,
    initial_layout=[0, 1, 2],
    optimization_level=3,
    seed_transpiler=42
)

def circuit_metrics(circuit):
    counts = circuit.count_ops()
    return {
        "depth": circuit.depth(),
        "total_gates": sum(counts.values()),
        "swap_gates": counts.get("swap", 0),
        "gate_counts": dict(counts)
    }

print("Optimization level 0:")
print(circuit_metrics(routed_level_0))

print("\nOptimization level 3:")
print(circuit_metrics(routed_level_3))

Optimization level 0:
{'depth': 5, 'total_gates': 6, 'swap_gates': 1, 'gate_counts': {'h': 2, 'cx': 2, 't': 1, 'swap': 1}}

Optimization level 3:
{'depth': 5, 'total_gates': 6, 'swap_gates': 1, 'gate_counts': {'h': 2, 'cx': 2, 't': 1, 'swap': 1}}


In [15]:
# Step 14: Verify both routed circuits

from qiskit.quantum_info import Operator
import numpy as np

op_level_0 = Operator.from_circuit(
    routed_level_0,
    ignore_set_layout=False
)

op_level_3 = Operator.from_circuit(
    routed_level_3,
    ignore_set_layout=False
)

print(
    "Level 0 equivalent to original:",
    np.allclose(Operator(qc).data, op_level_0.data)
)

print(
    "Level 3 equivalent to original:",
    np.allclose(Operator(qc).data, op_level_3.data)
)

Level 0 equivalent to original: True
Level 3 equivalent to original: True


In [16]:
# Step 15: Transpile to a specified gate basis

basis_qc = transpile(
    qc,
    basis_gates=["rz", "sx", "x", "cx"],
    coupling_map=coupling_map,
    initial_layout=[0, 1, 2],
    optimization_level=0,
    seed_transpiler=42
)

print("Circuit using the specified basis:")
print(basis_qc.draw())

print("\nGate counts:")
print(basis_qc.count_ops())

print("\nCircuit depth:", basis_qc.depth())

Circuit using the specified basis:
global phase: 5π/8
         ┌─────────┐┌────┐┌─────────┐                                »
q_0 -> 0 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├───────■────────────────────────»
         ├─────────┤└────┘└──┬───┬──┘     ┌─┴─┐┌───┐┌─────────┐┌────┐»
q_1 -> 1 ┤ Rz(π/4) ├──■──────┤ X ├─────■──┤ X ├┤ X ├┤ Rz(π/2) ├┤ √X ├»
         └─────────┘┌─┴─┐    └─┬─┘   ┌─┴─┐└───┘└─┬─┘└─────────┘└────┘»
q_2 -> 2 ───────────┤ X ├──────■─────┤ X ├───────■───────────────────»
                    └───┘            └───┘                           »
«                    
«q_0 -> 0 ───────────
«         ┌─────────┐
«q_1 -> 1 ┤ Rz(π/2) ├
«         └─────────┘
«q_2 -> 2 ───────────
«                    

Gate counts:
OrderedDict({'rz': 5, 'cx': 5, 'sx': 2})

Circuit depth: 9


In [17]:
# Step 16: Verify basis-gate circuit equivalence

from qiskit.quantum_info import Operator
import numpy as np

basis_operator = Operator.from_circuit(
    basis_qc,
    ignore_set_layout=False
)

original_operator = Operator(qc)

print(
    "Basis-gate circuit equivalent to original:",
    np.allclose(
        original_operator.data,
        basis_operator.data
    )
)

Basis-gate circuit equivalent to original: True


## Day 9 — Transpilation and Backend Optimization

### Experiments completed
- Built and measured a three-qubit circuit.
- Compared optimization levels 0, 1, 2, and 3.
- Created a linear coupling map.
- Observed routing overhead from one SWAP gate.
- Compared routed circuits at optimization levels 0 and 3.
- Verified logical-operation equivalence with layout accounted for.
- Transpiled to the basis gates rz, sx, x, and cx.
- Verified basis-gate circuit equivalence.

### Key observations
- Original circuit: 5 gates, depth 4.
- Routed circuit at optimization level 0: 6 gates, depth 5.
- Routing overhead: 20% more gates and 25% greater depth.
- Basis-gate circuit: 12 gates, depth 9.
- All layout-aware equivalence checks returned True.

### Conclusion
Transpilation adapts quantum circuits to target constraints.
Routing may introduce additional gates, and basis decomposition
may increase circuit size and depth. Verification is essential.